In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/sample_submission.csv
/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/species_mapping.csv
/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/train.csv
/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof/test.csv


In [2]:
# ============================================================
# XGB + CatBoost ensemble + correlation-aware meta + species-aware pseudo-labeling
# Focus improvements on:
#   - Amoxicillin_Clavulanic_acid (high missing): species-aware pseudo-labeling using teacher model
#   - Levofloxacin (and Cipro): correlation-aware meta + imbalance weights
# Includes:
#   - per-fold AUC logs
#   - OOF AUC per model
#   - optimal blend weight per target (grid)
#   - overfit summary (train vs val)
#   - ROC + calibration plots
#   - bootstrap CI for macro-AUC
# ============================================================

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from collections import defaultdict

import xgboost as xgb
from catboost import CatBoostClassifier

# -------------------
# CONFIG
# -------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

N_FOLDS_DEFAULT = 5
BOOTSTRAP_ITERS = 250

DATA_DIR = "/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test.csv")
SUB_PATH   = os.path.join(DATA_DIR, "sample_submission.csv")

ID_COL = "sample_id"
SPECIES_COL = "species_id"

TARGETS = [
    "Ampicillin",
    "Levofloxacin",
    "Ciprofloxacin",
    "Imipenem",
    "Amoxicillin_Clavulanic_acid",
    "Ertapenem",
    "Cefotaxime",
    "Cefuroxime"
]

# Where we *allow* pseudo-labeling
PSEUDO_TARGETS = {"Amoxicillin_Clavulanic_acid"}  # you can add "Levofloxacin" if it helps later

# pseudo-label settings (conservative by default)
PSEUDO_POS_THR = 0.985
PSEUDO_NEG_THR = 0.015
PSEUDO_MAX_PER_SPECIES_PER_CLASS = 20   # species-aware cap (key!)
PSEUDO_WEIGHT_BASE = 0.25               # downweight pseudo-labels
PSEUDO_MIN_KEEP = 20                    # if we keep < this, it's probably not worth using
PSEUDO_TEMP = 3.0                       # teacher sharpening: p -> sigmoid(logit(p)*TEMP)

# correlation-aware meta (we will use 2 correlated targets as "teacher features")
TOPK_CORR = 2
META_CV_FOLDS = 3

# output dirs
os.makedirs("plots_eval", exist_ok=True)
os.makedirs("models_eval", exist_ok=True)

# -------------------
# LOAD
# -------------------
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sub   = pd.read_csv(SUB_PATH)

spectrum_cols = [c for c in train.columns if c not in [ID_COL, SPECIES_COL] + TARGETS]

# -----------------------
# Preprocessing: log1p + global spectrum stats
# -----------------------
Xtr_spec = train[spectrum_cols].astype(np.float32).fillna(0.0)
Xte_spec = test[spectrum_cols].astype(np.float32).fillna(0.0)

if min(Xtr_spec.min().min(), Xte_spec.min().min()) >= 0:
    Xtr_spec = np.log1p(Xtr_spec)
    Xte_spec = np.log1p(Xte_spec)

def spectrum_stats(df):
    arr = df.to_numpy(dtype=np.float32)
    eps = 1e-12
    return pd.DataFrame({
        "spec_l1": np.sum(np.abs(arr), axis=1),
        "spec_l2": np.sqrt(np.sum(arr * arr, axis=1) + eps),
        "spec_max": np.max(arr, axis=1),
        "spec_mean": np.mean(arr, axis=1),
        "spec_std": np.std(arr, axis=1),
        "spec_nnz": np.sum(arr > 0, axis=1),
    }, index=df.index)

Xtr_stats = spectrum_stats(Xtr_spec)
Xte_stats = spectrum_stats(Xte_spec)

Xtr_num = pd.concat([Xtr_spec, Xtr_stats], axis=1)
Xte_num = pd.concat([Xte_spec, Xte_stats], axis=1)

# XGB: one-hot species
sp_tr = pd.get_dummies(train[SPECIES_COL], prefix="sp", dummy_na=True)
sp_te = pd.get_dummies(test[SPECIES_COL], prefix="sp", dummy_na=True).reindex(columns=sp_tr.columns, fill_value=0)
Xtr_xgb_all = pd.concat([Xtr_num, sp_tr], axis=1)
Xte_xgb_all = pd.concat([Xte_num, sp_te], axis=1)

# CatBoost: keep species categorical
Xtr_cat_all = Xtr_num.copy()
Xte_cat_all = Xte_num.copy()
Xtr_cat_all[SPECIES_COL] = train[SPECIES_COL].astype(str).fillna("NA")
Xte_cat_all[SPECIES_COL] = test[SPECIES_COL].astype(str).fillna("NA")
cat_features = [Xtr_cat_all.columns.get_loc(SPECIES_COL)]

# -------------------
# Helpers
# -------------------
def make_strat_key(species_series, y):
    # stratify by (species, label) -> better stability, more private-LB robust
    return species_series.astype(str).values + "_" + y.astype(str).values

def safe_n_folds(y, default=5):
    # avoid the "least populated class has only 1 members" failure
    # choose folds <= min_class_count (at least 2)
    y = np.asarray(y)
    c0 = np.sum(y == 0)
    c1 = np.sum(y == 1)
    m = int(min(c0, c1))
    if m < 2:
        return 2
    return int(min(default, m))

def logit(p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

def sharpen(p, temp=2.0):
    return sigmoid(logit(p) * temp)

def compute_class_weights(y):
    # returns per-sample weights (balanced)
    y = np.asarray(y)
    pos = np.sum(y == 1)
    neg = np.sum(y == 0)
    if pos == 0 or neg == 0:
        return np.ones_like(y, dtype=float), 1.0
    w_pos = neg / max(pos, 1.0)
    w = np.where(y == 1, w_pos, 1.0).astype(float)
    return w, w_pos

def best_blend_weight(y, p1, p2, grid=101):
    best_w, best_auc = 0.5, -1
    for w in np.linspace(0, 1, grid):
        ens = w*p1 + (1-w)*p2
        a = roc_auc_score(y, ens)
        if a > best_auc:
            best_auc = a
            best_w = float(w)
    return best_w, float(best_auc)

# -------------------
# Compute label correlations (Pearson on overlapping known rows)
# -------------------
corr_map = {}
print("Missing rates per target:")
missing_rates = {}
for t in TARGETS:
    missing = float(train[t].isna().mean())
    missing_rates[t] = missing
    print(f"  {t:28s} missing={missing:.3f}")

print("\nTop correlated targets (from labels, abs Pearson on overlapping known rows):")
for t in TARGETS:
    base = train[t]
    if base.notna().sum() < 50:
        corr_map[t] = []
        print(f"  {t:28s} -> []")
        continue

    corrs = []
    for u in TARGETS:
        if u == t:
            continue
        a = train[t]
        b = train[u]
        mask = a.notna() & b.notna()
        if mask.sum() < 100:
            continue
        r = np.corrcoef(a[mask].astype(float), b[mask].astype(float))[0,1]
        if np.isnan(r):
            continue
        corrs.append((u, abs(r)))
    corrs = sorted(corrs, key=lambda x: x[1], reverse=True)[:TOPK_CORR]
    corr_map[t] = [u for u,_ in corrs]
    print(f"  {t:28s} -> {corr_map[t]}")

# -------------------
# Model params (stronger imbalance handling + stability)
# -------------------
XGB_PARAMS_BASE = dict(
    n_estimators=3500,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=5,
    gamma=0.5,
    subsample=0.85,
    colsample_bytree=0.85,
    colsample_bynode=0.8,
    reg_lambda=2.0,
    reg_alpha=0.0,
    max_delta_step=1,        # helps imbalance
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    max_bin=256,
    random_state=SEED,
    n_jobs=-1,
    early_stopping_rounds=150
)

CAT_PARAMS_BASE = dict(
    iterations=6000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=8.0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    od_type="Iter",
    od_wait=250,
    verbose=False,
    allow_writing_files=False,
    # key improvement for imbalance:
    auto_class_weights="Balanced"
)

# -------------------
# 1) First pass: train base XGB+CAT for every target
#    - store OOF preds for correlation-aware meta + pseudo teachers
# -------------------
oof_xgb = {}
oof_cat = {}
oof_y = {}
oof_species = {}

test_pred_xgb = {t: np.zeros(len(test), dtype=float) for t in TARGETS}
test_pred_cat = {t: np.zeros(len(test), dtype=float) for t in TARGETS}

overfit_records = []

for tgt in TARGETS:
    print(f"\n==== {tgt} ====")
    y_series = train[tgt]
    mask_l = y_series.notna().values
    y_l = y_series.loc[mask_l].astype(int).values
    sp_l = train.loc[mask_l, SPECIES_COL].astype(str).values

    oof_y[tgt] = y_l
    oof_species[tgt] = sp_l

    if len(np.unique(y_l)) < 2:
        const = float(np.mean(y_l)) if len(y_l) else 0.5
        oof_xgb[tgt] = np.full(len(y_l), const)
        oof_cat[tgt] = np.full(len(y_l), const)
        test_pred_xgb[tgt][:] = const
        test_pred_cat[tgt][:] = const
        print("  Only one class in labeled rows -> constant preds")
        continue

    n_folds = safe_n_folds(y_l, default=N_FOLDS_DEFAULT)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    strat_key = make_strat_key(pd.Series(sp_l), pd.Series(y_l))

    X_l_xgb = Xtr_xgb_all.loc[mask_l].reset_index(drop=True)
    X_l_cat = Xtr_cat_all.loc[mask_l].reset_index(drop=True)

    oof_p_xgb = np.zeros(len(y_l), dtype=float)
    oof_p_cat = np.zeros(len(y_l), dtype=float)

    fold_test_xgb = np.zeros((len(test), n_folds), dtype=float)
    fold_test_cat = np.zeros((len(test), n_folds), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_l_xgb, strat_key), start=1):
        X_tr_xgb, X_va_xgb = X_l_xgb.iloc[tr_idx], X_l_xgb.iloc[va_idx]
        X_tr_cat, X_va_cat = X_l_cat.iloc[tr_idx], X_l_cat.iloc[va_idx]
        y_tr, y_va = y_l[tr_idx], y_l[va_idx]

        # imbalance weights
        w_tr, spw = compute_class_weights(y_tr)

        # ---- XGB
        xgb_params = dict(XGB_PARAMS_BASE)
        xgb_params["scale_pos_weight"] = spw
        m_xgb = xgb.XGBClassifier(**xgb_params)
        m_xgb.fit(X_tr_xgb, y_tr, sample_weight=w_tr, eval_set=[(X_va_xgb, y_va)], verbose=False)

        p_va_xgb = m_xgb.predict_proba(X_va_xgb)[:, 1]
        oof_p_xgb[va_idx] = p_va_xgb
        fold_test_xgb[:, fold-1] = m_xgb.predict_proba(Xte_xgb_all)[:, 1]

        # overfit check
        p_tr_xgb = m_xgb.predict_proba(X_tr_xgb)[:, 1]
        auc_tr_xgb = roc_auc_score(y_tr, p_tr_xgb)
        auc_va_xgb = roc_auc_score(y_va, p_va_xgb)

        # ---- CAT
        m_cat = CatBoostClassifier(**CAT_PARAMS_BASE)
        m_cat.fit(
            X_tr_cat, y_tr,
            sample_weight=w_tr,
            eval_set=(X_va_cat, y_va),
            cat_features=cat_features,
            use_best_model=True
        )
        p_va_cat = m_cat.predict_proba(X_va_cat)[:, 1]
        oof_p_cat[va_idx] = p_va_cat
        fold_test_cat[:, fold-1] = m_cat.predict_proba(Xte_cat_all)[:, 1]

        p_tr_cat = m_cat.predict_proba(X_tr_cat)[:, 1]
        auc_tr_cat = roc_auc_score(y_tr, p_tr_cat)
        auc_va_cat = roc_auc_score(y_va, p_va_cat)

        overfit_records.append({
            "target": tgt, "fold": fold,
            "auc_train_xgb": auc_tr_xgb, "auc_val_xgb": auc_va_xgb,
            "auc_train_cat": auc_tr_cat, "auc_val_cat": auc_va_cat,
            "delta_xgb": auc_tr_xgb - auc_va_xgb,
            "delta_cat": auc_tr_cat - auc_va_cat
        })

        print(f"  [fold {fold}/{n_folds}] XGB val={auc_va_xgb:.4f} | CAT val={auc_va_cat:.4f}")

    oof_xgb[tgt] = oof_p_xgb
    oof_cat[tgt] = oof_p_cat
    test_pred_xgb[tgt] = fold_test_xgb.mean(axis=1)
    test_pred_cat[tgt] = fold_test_cat.mean(axis=1)

# -------------------
# 2) Build correlation-aware meta models (using OOF preds of correlated targets)
#    These are *only trained on rows where both target and correlated labels exist*.
#    Output: meta_blend_preds (OOF for analysis, plus test-time meta predictions).
# -------------------
meta_used = {t: False for t in TARGETS}
meta_test_pred = {t: None for t in TARGETS}
meta_overlap_auc = {t: np.nan for t in TARGETS}
meta_blend_w = {t: np.nan for t in TARGETS}

def fit_meta_lr(tgt, corr_feats):
    # Build meta dataset on overlapping known rows for tgt and corr_feats
    y_t = train[tgt]
    mask = y_t.notna()
    for c in corr_feats:
        mask &= train[c].notna()

    idx = np.where(mask.values)[0]
    if len(idx) < 300:
        return None  # too small, skip

    y = train.loc[idx, tgt].astype(int).values

    # meta features: OOF preds of correlated targets (using their OOF preds aligned to these idx)
    X_parts = []
    for c in corr_feats:
        # build OOF aligned array for c:
        # oof_*[c] exists only on labeled rows of c -> map by index
        # easiest: recompute via a mask mapping
        mask_c = train[c].notna().values
        idx_c = np.where(mask_c)[0]
        # create full-length buffer, fill only labeled positions for c
        full = np.full(len(train), np.nan, dtype=float)
        full[idx_c] = oof_cat[c]  # cat OOF is slightly stronger in your logs; use CAT by default
        X_parts.append(full[idx])

    X = np.vstack(X_parts).T
    if np.isnan(X).any():
        return None

    # quick CV to estimate overlap AUC and fit final model
    skf = StratifiedKFold(n_splits=META_CV_FOLDS, shuffle=True, random_state=SEED)
    oof_meta = np.zeros(len(y), dtype=float)

    for tr, va in skf.split(X, y):
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X[tr])
        X_va = scaler.transform(X[va])
        lr = LogisticRegression(max_iter=2000, n_jobs=-1)
        lr.fit(X_tr, y[tr])
        oof_meta[va] = lr.predict_proba(X_va)[:, 1]

    auc = roc_auc_score(y, oof_meta)

    # fit final on all overlap rows
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    lr = LogisticRegression(max_iter=2000, n_jobs=-1)
    lr.fit(Xs, y)

    # make test meta pred
    # build test features as mean test preds of correlated targets
    Xt = np.vstack([test_pred_cat[c] for c in corr_feats]).T
    Xt = scaler.transform(Xt)
    p_test = lr.predict_proba(Xt)[:, 1]

    return auc, p_test

print("\n=== CORRELATION-AWARE META (teacher + final blend candidate) ===")
for tgt in TARGETS:
    corr_feats = corr_map.get(tgt, [])
    if len(corr_feats) == 0:
        continue
    res = fit_meta_lr(tgt, corr_feats)
    if res is None:
        continue
    auc, p_test = res
    meta_used[tgt] = True
    meta_test_pred[tgt] = p_test
    meta_overlap_auc[tgt] = auc
    print(f"[META] {tgt}: used corr={corr_feats} overlap_auc={auc:.4f} rows>=300")

# -------------------
# 3) Species-aware pseudo-labeling for Amoxicillin_Clavulanic_acid (and any in PSEUDO_TARGETS)
#    Teacher = correlation-meta prediction if available (preferred), else ensemble base prediction
#    Pick per-species top confident positives/negatives, capped.
#    Retrain XGB+CAT inside each fold using pseudo-labeled *unlabeled pool*.
# -------------------
def species_aware_pseudo_idx(species_u, p_u, pos_thr, neg_thr, max_per_species_per_class):
    # returns keep_idx and pseudo_y
    keep = []
    yps = []

    species_u = np.asarray(species_u).astype(str)
    p_u = np.asarray(p_u)

    for sp in np.unique(species_u):
        idx = np.where(species_u == sp)[0]
        if len(idx) == 0:
            continue
        ps = p_u[idx]

        pos = idx[ps >= pos_thr]
        neg = idx[ps <= neg_thr]

        # sort by confidence
        pos = pos[np.argsort(-p_u[pos])]
        neg = neg[np.argsort(p_u[neg])]

        pos = pos[:max_per_species_per_class]
        neg = neg[:max_per_species_per_class]

        keep.extend(pos.tolist())
        yps.extend([1]*len(pos))
        keep.extend(neg.tolist())
        yps.extend([0]*len(neg))

    keep = np.array(keep, dtype=int)
    yps = np.array(yps, dtype=int)
    return keep, yps

def run_target_with_optional_pseudo(tgt):
    y_series = train[tgt]
    mask_l = y_series.notna().values
    y_l = y_series.loc[mask_l].astype(int).values
    sp_l = train.loc[mask_l, SPECIES_COL].astype(str).values

    if len(np.unique(y_l)) < 2:
        const = float(np.mean(y_l)) if len(y_l) else 0.5
        return dict(
            oof_auc_xgb=np.nan, oof_auc_cat=np.nan, best_w_xgb=0.5, oof_auc_ens=np.nan,
            test_pred=np.full(len(test), const),
            pseudo_used=False
        )

    n_folds = safe_n_folds(y_l, default=N_FOLDS_DEFAULT)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    strat_key = make_strat_key(pd.Series(sp_l), pd.Series(y_l))

    X_l_xgb = Xtr_xgb_all.loc[mask_l].reset_index(drop=True)
    X_l_cat = Xtr_cat_all.loc[mask_l].reset_index(drop=True)

    # unlabeled pool (for this target)
    mask_u = ~mask_l
    X_u_xgb = Xtr_xgb_all.loc[mask_u].reset_index(drop=True)
    X_u_cat = Xtr_cat_all.loc[mask_u].reset_index(drop=True)
    sp_u = train.loc[mask_u, SPECIES_COL].astype(str).values

    # choose teacher probabilities for unlabeled:
    # prefer meta_test_pred-like idea BUT we need teacher for *train unlabeled*, not test.
    # So we compute teacher on unlabeled rows from correlated targets' OOF preds (if meta possible).
    corr_feats = corr_map.get(tgt, [])
    p_teacher_u = None
    if len(corr_feats) > 0:
        # build teacher features using correlated targets OOF predictions aligned to full train rows
        mask_overlap = mask_u.copy()
        ok = True
        X_parts = []
        for c in corr_feats:
            mask_c = train[c].notna().values
            idx_c = np.where(mask_c)[0]
            full = np.full(len(train), np.nan, dtype=float)
            full[idx_c] = oof_cat[c]  # correlated target oof preds
            X_parts.append(full[mask_u])
            if np.isnan(X_parts[-1]).any():
                ok = False
        if ok:
            Xu = np.vstack(X_parts).T
            # fit teacher LR on overlap rows where tgt and corr feats all known
            y_t = train[tgt]
            mask_meta = y_t.notna()
            for c in corr_feats:
                mask_meta &= train[c].notna()
            idx_meta = np.where(mask_meta.values)[0]
            if len(idx_meta) >= 300:
                y_meta = train.loc[idx_meta, tgt].astype(int).values
                Xm_parts = []
                for c in corr_feats:
                    mask_c = train[c].notna().values
                    idx_c = np.where(mask_c)[0]
                    full = np.full(len(train), np.nan, dtype=float)
                    full[idx_c] = oof_cat[c]
                    Xm_parts.append(full[idx_meta])
                Xm = np.vstack(Xm_parts).T
                if not np.isnan(Xm).any():
                    scaler = StandardScaler()
                    Xm_s = scaler.fit_transform(Xm)
                    lr = LogisticRegression(max_iter=2000, n_jobs=-1)
                    lr.fit(Xm_s, y_meta)
                    p_teacher_u = lr.predict_proba(scaler.transform(Xu))[:, 1]
                    p_teacher_u = sharpen(p_teacher_u, temp=PSEUDO_TEMP)

    # fallback teacher if meta not possible: use base ensemble prediction from first pass
    if p_teacher_u is None:
        # build base teacher on unlabeled using base ensemble (cat is stronger)
        # We can get base teacher preds for ALL train rows if we re-fit target once; too expensive.
        # So here we do "no pseudo" unless meta teacher exists.
        return None  # signal "skip pseudo path"

    # prepare storage
    oof_p_xgb = np.zeros(len(y_l), dtype=float)
    oof_p_cat = np.zeros(len(y_l), dtype=float)

    test_fold_xgb = np.zeros((len(test), n_folds), dtype=float)
    test_fold_cat = np.zeros((len(test), n_folds), dtype=float)

    pseudo_kept_total = 0
    pseudo_used_any = False

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_l_xgb, strat_key), start=1):
        X_tr_xgb, X_va_xgb = X_l_xgb.iloc[tr_idx], X_l_xgb.iloc[va_idx]
        X_tr_cat, X_va_cat = X_l_cat.iloc[tr_idx], X_l_cat.iloc[va_idx]
        y_tr, y_va = y_l[tr_idx], y_l[va_idx]

        # base weights
        w_tr, spw = compute_class_weights(y_tr)

        # --- pseudo selection (species-aware)
        keep_u, y_u = species_aware_pseudo_idx(
            species_u=sp_u,
            p_u=p_teacher_u,
            pos_thr=PSEUDO_POS_THR,
            neg_thr=PSEUDO_NEG_THR,
            max_per_species_per_class=PSEUDO_MAX_PER_SPECIES_PER_CLASS
        )

        kept = len(keep_u)
        pseudo_kept_total += kept

        # if we don't keep enough, do not use pseudo for this fold
        use_pseudo = kept >= PSEUDO_MIN_KEEP
        pseudo_used_any = pseudo_used_any or use_pseudo

        if use_pseudo:
            X_pl_xgb = X_u_xgb.iloc[keep_u]
            X_pl_cat = X_u_cat.iloc[keep_u]

            # pseudo weights: base * confidence
            conf = np.abs(p_teacher_u[keep_u] - 0.5) * 2.0  # [0,1]
            w_pseudo = PSEUDO_WEIGHT_BASE * (0.25 + 0.75*conf)

            X_aug_xgb = pd.concat([X_tr_xgb, X_pl_xgb], axis=0)
            X_aug_cat = pd.concat([X_tr_cat, X_pl_cat], axis=0)
            y_aug = np.concatenate([y_tr, y_u], axis=0)

            # combine weights: class weights for real labels + pseudo weights
            w_aug_real, spw_aug = compute_class_weights(y_tr)
            w_aug = np.concatenate([w_aug_real, w_pseudo], axis=0)

        # ---- XGB
        xgb_params = dict(XGB_PARAMS_BASE)
        xgb_params["scale_pos_weight"] = spw

        m_xgb = xgb.XGBClassifier(**xgb_params)
        if use_pseudo:
            m_xgb.fit(X_aug_xgb, y_aug, sample_weight=w_aug, eval_set=[(X_va_xgb, y_va)], verbose=False)
        else:
            m_xgb.fit(X_tr_xgb, y_tr, sample_weight=w_tr, eval_set=[(X_va_xgb, y_va)], verbose=False)

        p_va_xgb = m_xgb.predict_proba(X_va_xgb)[:, 1]
        oof_p_xgb[va_idx] = p_va_xgb
        test_fold_xgb[:, fold-1] = m_xgb.predict_proba(Xte_xgb_all)[:, 1]

        # ---- CAT
        m_cat = CatBoostClassifier(**CAT_PARAMS_BASE)
        if use_pseudo:
            m_cat.fit(
                X_aug_cat, y_aug,
                sample_weight=w_aug,
                eval_set=(X_va_cat, y_va),
                cat_features=cat_features,
                use_best_model=True
            )
        else:
            m_cat.fit(
                X_tr_cat, y_tr,
                sample_weight=w_tr,
                eval_set=(X_va_cat, y_va),
                cat_features=cat_features,
                use_best_model=True
            )

        p_va_cat = m_cat.predict_proba(X_va_cat)[:, 1]
        oof_p_cat[va_idx] = p_va_cat
        test_fold_cat[:, fold-1] = m_cat.predict_proba(Xte_cat_all)[:, 1]

        # fold logs
        auc_x = roc_auc_score(y_va, p_va_xgb)
        auc_c = roc_auc_score(y_va, p_va_cat)
        print(f"  [fold {fold}/{n_folds}] XGB val={auc_x:.4f} | CAT val={auc_c:.4f} | pseudo={use_pseudo} kept={kept}")

    # target-level AUCs
    auc_xgb = float(roc_auc_score(y_l, oof_p_xgb))
    auc_cat = float(roc_auc_score(y_l, oof_p_cat))

    # ensemble best blend
    best_w, auc_ens = best_blend_weight(y_l, oof_p_xgb, oof_p_cat, grid=101)

    # test preds
    p_test_xgb = test_fold_xgb.mean(axis=1)
    p_test_cat = test_fold_cat.mean(axis=1)
    p_test = best_w*p_test_xgb + (1-best_w)*p_test_cat

    pseudo_used = (pseudo_used_any and pseudo_kept_total >= PSEUDO_MIN_KEEP)
    return dict(
        oof_auc_xgb=auc_xgb,
        oof_auc_cat=auc_cat,
        best_w_xgb=best_w,
        oof_auc_ens=auc_ens,
        test_pred=p_test,
        pseudo_used=pseudo_used
    )

# -------------------
# 4) Final per-target decision:
#    - if tgt in PSEUDO_TARGETS and meta teacher exists -> retrain with species-aware pseudo
#    - else -> use base first-pass (already done) with best blend weight
#    - then optional meta-blending for correlated ones (Levo, Cefotaxime, Amox)
# -------------------
final_test_pred = pd.DataFrame({ID_COL: test[ID_COL].values})
summary_rows = []

for tgt in TARGETS:
    print(f"\n==== FINAL {tgt} ====")
    y_l = oof_y[tgt]
    if len(np.unique(y_l)) < 2:
        const = float(np.mean(y_l)) if len(y_l) else 0.5
        final_test_pred[tgt] = const
        summary_rows.append(dict(
            target=tgt, missing_rate=missing_rates[tgt],
            oof_auc_xgb=np.nan, oof_auc_cat=np.nan, best_w_xgb=0.5, oof_auc_ens=np.nan,
            meta_used=False, meta_blend_w=np.nan, meta_overlap_auc=np.nan
        ))
        continue

    # base ensemble on OOF from first pass
    w_base, auc_base = best_blend_weight(y_l, oof_xgb[tgt], oof_cat[tgt], grid=101)
    p_base_test = w_base*test_pred_xgb[tgt] + (1-w_base)*test_pred_cat[tgt]

    # optionally rerun pseudo path
    pseudo_used = False
    if tgt in PSEUDO_TARGETS:
        out = run_target_with_optional_pseudo(tgt)
        if out is not None and out["pseudo_used"]:
            pseudo_used = True
            # replace base with pseudo-trained model outputs
            w_base, auc_base = out["best_w_xgb"], out["oof_auc_ens"]
            p_base_test = out["test_pred"]
            print(f"  --> using pseudo-enhanced training for {tgt} (OOF ens={auc_base:.4f})")
        else:
            print(f"  --> pseudo teacher not available or not helpful for {tgt}; keeping base")

    # meta blend (if meta exists for this target)
    used_meta = False
    mbw = np.nan
    moauc = np.nan
    p_final = p_base_test

    if meta_used.get(tgt, False) and meta_test_pred[tgt] is not None:
        # choose blend weight for meta using overlapping rows only
        # (we use overlap_auc as signal, and do a conservative blend)
        moauc = float(meta_overlap_auc[tgt])
        # conservative rule: if meta overlap AUC > base OOF ens AUC by margin -> use more meta
        # else tiny blend
        if moauc > auc_base + 0.002:
            mbw = 0.65
        else:
            mbw = 0.25
        p_final = (1-mbw)*p_base_test + mbw*meta_test_pred[tgt]
        used_meta = True
        print(f"  --> meta blend used for {tgt}: meta_w={mbw:.2f} (overlap_auc={moauc:.4f}, base_oof={auc_base:.4f})")

    final_test_pred[tgt] = p_final

    summary_rows.append(dict(
        target=tgt,
        missing_rate=missing_rates[tgt],
        oof_auc_xgb=float(roc_auc_score(y_l, oof_xgb[tgt])),
        oof_auc_cat=float(roc_auc_score(y_l, oof_cat[tgt])),
        best_w_xgb=w_base,
        oof_auc_ens=auc_base,
        meta_used=used_meta,
        meta_blend_w=mbw,
        meta_overlap_auc=moauc,
        pseudo_used=pseudo_used
    ))

# -------------------
# 5) Save submission
# -------------------
submission = sub[[ID_COL]].merge(final_test_pred, on=ID_COL, how="left")
submission.to_csv("submission.csv", index=False)
print("\nSaved submission.csv")
display(submission.head())

# -------------------
# 6) Evaluation artifacts: ROC + calibration (OOF for base ensemble)
#     NOTE: For pseudo targets, OOF changed only if pseudo applied; here we plot base OOF ensemble.
# -------------------
for tgt in TARGETS:
    y = oof_y[tgt]
    if len(np.unique(y)) < 2:
        continue
    w, auc_ens = best_blend_weight(y, oof_xgb[tgt], oof_cat[tgt], grid=101)
    ens = w*oof_xgb[tgt] + (1-w)*oof_cat[tgt]

    # ROC
    RocCurveDisplay.from_predictions(y, ens)
    plt.title(f"ROC OOF - {tgt} (ENS AUC {auc_ens:.3f})")
    plt.savefig(f"plots_eval/roc_oof_{tgt}.png", dpi=160, bbox_inches="tight")
    plt.close()

    # Calibration
    prob_true, prob_pred = calibration_curve(y, ens, n_bins=10)
    plt.figure(figsize=(5,5))
    plt.plot(prob_pred, prob_true, marker="o", label="ENS")
    plt.plot([0,1],[0,1],"--", color="gray")
    plt.title(f"Calibration OOF - {tgt}")
    plt.legend()
    plt.savefig(f"plots_eval/calibration_oof_{tgt}.png", dpi=160, bbox_inches="tight")
    plt.close()

print("\nSaved ROC + calibration plots to plots_eval/")

# -------------------
# 7) Overfitting summary
# -------------------
overfit_df = pd.DataFrame(overfit_records)
if len(overfit_df) > 0:
    ov = overfit_df.groupby("target")[["delta_xgb","delta_cat"]].mean().sort_values("delta_cat", ascending=False)
    ov.to_csv("plots_eval/overfitting_summary.csv")
    print("\nOverfitting summary saved to plots_eval/overfitting_summary.csv")
    display(ov)

# -------------------
# 8) Bootstrap CI for macro-AUC (base ensemble OOF)
# -------------------
print("\nBootstrap CI (OOF opt-weight ensemble) ...")

def macro_auc_from_oof():
    aucs = []
    for tgt in TARGETS:
        y = oof_y[tgt]
        if len(np.unique(y)) < 2:
            continue
        w, _ = best_blend_weight(y, oof_xgb[tgt], oof_cat[tgt], grid=101)
        ens = w*oof_xgb[tgt] + (1-w)*oof_cat[tgt]
        aucs.append(roc_auc_score(y, ens))
    return float(np.mean(aucs))

base_macro = macro_auc_from_oof()

boot = []
for _ in range(BOOTSTRAP_ITERS):
    per_tgt = []
    for tgt in TARGETS:
        y = oof_y[tgt]
        if len(np.unique(y)) < 2:
            continue
        w, _ = best_blend_weight(y, oof_xgb[tgt], oof_cat[tgt], grid=101)
        ens = w*oof_xgb[tgt] + (1-w)*oof_cat[tgt]
        n = len(y)
        idx = np.random.choice(np.arange(n), size=n, replace=True)
        try:
            per_tgt.append(roc_auc_score(y[idx], ens[idx]))
        except:
            pass
    if len(per_tgt):
        boot.append(np.mean(per_tgt))

boot = np.array(boot)
ci_lo = float(np.percentile(boot, 2.5))
ci_hi = float(np.percentile(boot, 97.5))
print(f"\nMacro OOF AUC (opt-weight ensemble): {base_macro:.4f}")
print(f"Bootstrap Macro AUC: mean={boot.mean():.4f} 95%CI [{ci_lo:.4f}, {ci_hi:.4f}]")

# -------------------
# 9) Summary table
# -------------------
summary_df = pd.DataFrame(summary_rows).sort_values("oof_auc_ens")
print("\n=== TARGET SUMMARY ===")
display(summary_df)

summary_df.to_csv("plots_eval/target_summary.csv", index=False)
print("\nSaved plots_eval/target_summary.csv")


Missing rates per target:
  Ampicillin                   missing=0.008
  Levofloxacin                 missing=0.026
  Ciprofloxacin                missing=0.026
  Imipenem                     missing=0.033
  Amoxicillin_Clavulanic_acid  missing=0.428
  Ertapenem                    missing=0.000
  Cefotaxime                   missing=0.001
  Cefuroxime                   missing=0.010

Top correlated targets (from labels, abs Pearson on overlapping known rows):
  Ampicillin                   -> ['Cefotaxime', 'Ertapenem']
  Levofloxacin                 -> ['Ciprofloxacin', 'Cefotaxime']
  Ciprofloxacin                -> ['Levofloxacin', 'Cefotaxime']
  Imipenem                     -> ['Ertapenem', 'Cefotaxime']
  Amoxicillin_Clavulanic_acid  -> ['Cefuroxime', 'Cefotaxime']
  Ertapenem                    -> ['Cefotaxime', 'Imipenem']
  Cefotaxime                   -> ['Ertapenem', 'Cefuroxime']
  Cefuroxime                   -> ['Cefotaxime', 'Ertapenem']

==== Ampicillin ====


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


  [fold 1/5] XGB val=0.9185 | CAT val=0.9331
  [fold 2/5] XGB val=0.9170 | CAT val=0.9317
  [fold 3/5] XGB val=0.9228 | CAT val=0.9349
  [fold 4/5] XGB val=0.9240 | CAT val=0.9413
  [fold 5/5] XGB val=0.9215 | CAT val=0.9379

==== Levofloxacin ====
  [fold 1/5] XGB val=0.8435 | CAT val=0.8565
  [fold 2/5] XGB val=0.8247 | CAT val=0.8325
  [fold 3/5] XGB val=0.8668 | CAT val=0.8750
  [fold 4/5] XGB val=0.8425 | CAT val=0.8558
  [fold 5/5] XGB val=0.8528 | CAT val=0.8569

==== Ciprofloxacin ====
  [fold 1/5] XGB val=0.8463 | CAT val=0.8507
  [fold 2/5] XGB val=0.8383 | CAT val=0.8449
  [fold 3/5] XGB val=0.8467 | CAT val=0.8535
  [fold 4/5] XGB val=0.8576 | CAT val=0.8629
  [fold 5/5] XGB val=0.8342 | CAT val=0.8500

==== Imipenem ====


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


  [fold 1/5] XGB val=0.9898 | CAT val=0.9920
  [fold 2/5] XGB val=0.9929 | CAT val=0.9944
  [fold 3/5] XGB val=0.9920 | CAT val=0.9907
  [fold 4/5] XGB val=0.9889 | CAT val=0.9882
  [fold 5/5] XGB val=0.9952 | CAT val=0.9933

==== Amoxicillin_Clavulanic_acid ====
  [fold 1/5] XGB val=0.7034 | CAT val=0.7164
  [fold 2/5] XGB val=0.7455 | CAT val=0.7460
  [fold 3/5] XGB val=0.7562 | CAT val=0.7596
  [fold 4/5] XGB val=0.7313 | CAT val=0.7473
  [fold 5/5] XGB val=0.7215 | CAT val=0.7171

==== Ertapenem ====


...


Saved ROC + calibration plots to plots_eval/

Overfitting summary saved to plots_eval/overfitting_summary.csv


,delta_xgb,delta_cat
target,,
Amoxicillin_Clavulanic_acid,0.233979,0.255262
Ciprofloxacin,0.151238,0.142642
Levofloxacin,0.131451,0.132917
Cefotaxime,0.043988,0.054099
Cefuroxime,0.041578,0.048913
Ampicillin,0.004476,0.044271
Ertapenem,0.010566,0.009644
Imipenem,0.006075,0.003257



Bootstrap CI (OOF opt-weight ensemble) ...

Macro OOF AUC (opt-weight ensemble): 0.9039
Bootstrap Macro AUC: mean=0.9041 95%CI [0.8999, 0.9082]

=== TARGET SUMMARY ===


,target,missing_rate,oof_auc_xgb,oof_auc_cat,best_w_xgb,oof_auc_ens,meta_used,meta_blend_w,meta_overlap_auc,pseudo_used
4,Amoxicillin_Clavulanic_acid,0.428274,0.706284,0.730384,0.21,0.730974,True,0.25,0.696059,False
2,Ciprofloxacin,0.025595,0.843779,0.848452,0.38,0.850832,True,0.25,0.845133,False
1,Levofloxacin,0.026488,0.834318,0.855330,0.13,0.856099,True,0.25,0.847594,False
0,Ampicillin,0.008333,0.913580,0.932926,0.05,0.932934,True,0.25,0.848223,False
6,Cefotaxime,0.000893,0.930369,0.935351,0.25,0.935891,True,0.25,0.912056,False
7,Cefuroxime,0.009821,0.936388,0.947042,0.13,0.947139,True,0.25,0.854437,False
5,Ertapenem,0.000000,0.984021,0.982055,0.66,0.987155,True,0.25,0.981513,False
3,Imipenem,0.033036,0.989175,0.986716,0.82,0.990099,True,0.25,0.852806,False



Saved plots_eval/target_summary.csv


How to tune it safely (very important)
If pseudo-labeling helps:

--> keep it only for high-missing targets (like Amox/Clav)

--> maybe loosen slightly to 0.985 / 0.015

--> increase PSEUDO_MAX_PER_CLASS slowly (300 → 400)

If pseudo-labeling hurts:

--> reduce pseudo weight: 0.25 → 0.15

--> tighten thresholds: 0.995 / 0.005

--> reduce max per class: 300 → 150

For imbalance:

--> CatBoost auto_class_weights="Balanced" is already excellent

--> XGB: keep scale_pos_weight, and the extra per-row weights only matter for extreme imbalance (ratio ≥ 5)